<a href="https://colab.research.google.com/github/meghansn/glp1-analysis/blob/main/Data_Organization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import auth
auth.authenticate_user()

# Optional: Set your project ID to ensure the gcloud CLI knows which project to use
# Replace 'my-pharma-data-project' with your specific Google Cloud Project ID if different
project_id = 'my-pharma-data-project'
!gcloud config set project {project_id}

Are you sure you wish to set property [core/project] to my-pharma-data-project?

Do you want to continue (Y/n)?  y

Updated property [core/project].


In [2]:
!gcloud storage ls gs://my-pharma-data-project/

gs://my-pharma-data-project/adverse_events.csv
gs://my-pharma-data-project/adverse_events_summary.csv
gs://my-pharma-data-project/clinical_trials.csv
gs://my-pharma-data-project/data_dictionary.csv
gs://my-pharma-data-project/drugs_overview.csv
gs://my-pharma-data-project/search_trends.csv
gs://my-pharma-data-project/stock_prices.csv
gs://my-pharma-data-project/wikipedia_summaries.csv
gs://my-pharma-data-project/canada-adverse-data/


In [3]:
# The complete list of files for ML, MLOps, and RAG
all_ml_files = [
    # Canadian Adverse Data (already converted to CSV)
    'reports.csv', 'reactions.csv', 'report_drug.csv',
    'drug_products.csv', 'outcome_lx.csv', 'seriousness_lx.csv',
    # Kaggle/External Data
    'adverse_events.csv', 'adverse_events_summary.csv',
    'wikipedia_summaries.csv', 'clinical_trials.csv',
    'drugs_overview.csv', 'search_trends.csv', 'data_dictionary.csv'
]

source_prefix_canada = 'gs://my-pharma-data-project/canada-adverse-data/csv_format/'
source_prefix_root = 'gs://my-pharma-data-project/'
dest_prefix = 'gs://my-pharma-data-project/ml-project-data/'

print("Organizing all ML/RAG files into ml-project-data/ folder...")

for filename in all_ml_files:
    # Determine the source based on where the file lives
    if filename in ['reports.csv', 'reactions.csv', 'report_drug.csv', 'drug_products.csv', 'outcome_lx.csv', 'seriousness_lx.csv']:
        source_path = f"{source_prefix_canada}{filename}"
    else:
        source_path = f"{source_prefix_root}{filename}"

    dest_path = f"{dest_prefix}{filename}"

    print(f"Copying: {filename}")
    !gcloud storage cp {source_path} {dest_path}

print("\nSuccess! All ML/RAG files are now organized in ml-project-data/")

Organizing all ML/RAG files into ml-project-data/ folder...
Copying: reports.csv
Copying gs://my-pharma-data-project/canada-adverse-data/csv_format/reports.csv to gs://my-pharma-data-project/ml-project-data/reports.csv
Copying: reactions.csv
Copying gs://my-pharma-data-project/canada-adverse-data/csv_format/reactions.csv to gs://my-pharma-data-project/ml-project-data/reactions.csv
Copying: report_drug.csv
Copying gs://my-pharma-data-project/canada-adverse-data/csv_format/report_drug.csv to gs://my-pharma-data-project/ml-project-data/report_drug.csv
Copying: drug_products.csv
Copying gs://my-pharma-data-project/canada-adverse-data/csv_format/drug_products.csv to gs://my-pharma-data-project/ml-project-data/drug_products.csv
Copying: outcome_lx.csv
Copying gs://my-pharma-data-project/canada-adverse-data/csv_format/outcome_lx.csv to gs://my-pharma-data-project/ml-project-data/outcome_lx.csv
Copying: seriousness_lx.csv
Copying gs://my-pharma-data-project/canada-adverse-data/csv_format/serio

In [ ]:
import pandas as pd
import io

# The folder where we organized all your "Gold" ML/RAG files
target_folder = 'gs://my-pharma-data-project/ml-project-data/'

# 1. Get the list of all files in the folder
file_list_output = !gcloud storage ls {target_folder}
# Clean up the output to get just the paths
file_paths = [line.strip() for line in file_list_output if line.strip()]

print(f"Found {len(file_paths)} files. Previewing now...\n")

# 2. Loop through and preview each one
for path in file_paths:
    filename = path.split('/')[-1]
    print(f"\n{'='*20}\nPREVIEW: {filename}\n{'='*20}")

    try:
        # Use gcloud to stream the file content without downloading it locally
        # This is efficient for large datasets
        content = !gcloud storage cat {path}

        # Convert the command output (list of strings) into a single string
        # and then into a dataframe
        csv_data = "\n".join(content)
        df = pd.read_csv(io.StringIO(csv_data))

        display(df.head(5))
        print(f"Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")

    except Exception as e:
        print(f"Could not preview {filename}: {e}")

Found 13 files. Previewing now...


PREVIEW: adverse_events.csv


,safetyreportid,generic_name,brand_queried,receive_date,country,serious,seriousness_death,seriousness_lifethreatening,seriousness_hospitalization,seriousness_disabling,patient_age,patient_age_unit,patient_sex,patient_weight_kg,reaction,reaction_outcome
0,10188271,semaglutide,OZEMPIC,2014-05-22,US,True,False,False,True,True,81.0,Year,Female,NaN,Stenosis,Unknown
1,10188271,semaglutide,OZEMPIC,2014-05-22,US,True,False,False,True,True,81.0,Year,Female,NaN,Hunger,Unknown
2,10188271,semaglutide,OZEMPIC,2014-05-22,US,True,False,False,True,True,81.0,Year,Female,NaN,Visual impairment,Unknown
3,10188271,semaglutide,OZEMPIC,2014-05-22,US,True,False,False,True,True,81.0,Year,Female,NaN,Incorrect dose administered,Unknown
4,10188271,semaglutide,OZEMPIC,2014-05-22,US,True,False,False,True,True,81.0,Year,Female,NaN,Inappropriate schedule of product administration,Unknown


Dimensions: 149209 rows, 16 columns

PREVIEW: adverse_events_summary.csv


,generic_name,reaction,report_count,pct_serious,pct_hospitalization,pct_death,median_age,pct_female
0,albiglutide,Device use error,2699,0.0189,0.0052,0.0,58.0,0.5873
1,albiglutide,Accidental exposure to product,840,0.0202,0.0060,0.0,58.0,0.5274
2,albiglutide,Drug dose omission,662,0.0468,0.0211,0.0,59.0,0.6073
3,albiglutide,Device leakage,620,0.0210,0.0048,0.0,58.0,0.5387
4,albiglutide,Product quality issue,483,0.0248,0.0104,0.0,60.0,0.5507


Dimensions: 11093 rows, 8 columns

PREVIEW: clinical_trials.csv


...

Dimensions: 10 rows, 7 columns

PREVIEW: outcome_lx.csv


,"""1906""","""06""","""Unknown""","""Inconnue"""
0,"""1907""","""07""","""Recovered/resolved""","""GuÃ©rison"""
1,"""1908""","""08""","""Recovering/resolving""","""GuÃ©rison en cours"""
2,"""1909""","""09""","""Not recovered/not resolved""","""Non rÃ©tabli/Non rÃ©solu"""
3,"""1910""","""10""","""Recovered/resolved with sequelae""","""GuÃ©rison avec sÃ©quelles"""
4,"""1911""","""11""","""Fatal""","""Fatale"""


Dimensions: 5 rows, 4 columns

PREVIEW: reactions.csv


,"""5601""","""56""","""""",""""".1",""""".2","""Rash""","""Rash"".1","""Skin and subcutaneous tissue disorders""","""Affections de la peau et du tissu sous-cutanÃ©""","""v.27.1"""
0,"""5701""","""57""","""""","""""","""""","""Irritability""","""IrritabilitÃ©""","""Psychiatric disorders""","""Affections psychiatriques""","""v.27.1"""
1,"""5801""","""58""","""""","""""","""""","""Conjunctivitis""","""Conjonctivite""","""Infections and infestations""","""Infections et infestations""","""v.27.1"""
2,"""5803""","""58""","""""","""""","""""","""Photophobia""","""Photophobie""","""Eye disorders""","""Affections oculaires""","""v.27.1"""
3,"""5802""","""58""","""""","""""","""""","""Neuralgia""","""NÃ©vralgie""","""Nervous system disorders""","""Affections du systÃ¨me nerveux""","""v.27.1"""
4,"""5901""","""59""","""""","""""","""""","""Rash""","""Rash""","""Skin and subcutaneous tissue disorders""","""Affections de la peau et du tissu sous-cutanÃ©""","""v.27.1"""


Dimensions: 4474922 rows, 10 columns

PREVIEW: report_drug.csv
